# 📘 **Week 11: Text Cleaning and Preprocessing**

**Business Data and Text Mining — BA3002**  
**BS Business Analytics | Fall 2026 | Sections A and B**

---

**👨‍🏫 Dr. Usama Arshad**  
Assistant Professor (Business Analytics), FAST-NUCES Islamabad  
Director & Founder, CRADLE Lab — Computational Research in AI, Data Science, and Ledger Engineering  
[Website](https://usamajanjua.com) · [GitHub](https://github.com/usamajanjua9)

### 🎯 **Learning Goals**

- Design preprocessing choices around the business question and language.
- Clean text while preserving negation, useful quantities, and the original evidence.
- Compare sentence, word, and subword tokenization.
- Explain when stop-word removal, stemming, and lemmatization help or harm.
- Construct unigrams, bigrams, and trigrams without crossing document boundaries.
- Build, validate, and export a traceable corpus from the Week 10 review data.

### 🧭 **Learning Roadmap**

1. [Introduction to Text Preprocessing](#section-1)
2. [Text Cleaning and Normalization](#section-2)
3. [Tokenization](#section-3)
4. [Stop-Word Removal](#section-4)
5. [Stemming and Lemmatization](#section-5)
6. [N-Grams and Text Pattern Discovery](#section-6)
7. [Preparing a Text Corpus](#section-7)
8. [Business Case Studies and Error-Spotting Activities](#section-8)
9. [Summary and Reinforcement](#section-9)
10. [Additional Reading and Resources](#section-10)

[💻 Lab Week 11](#lab)

> **Learning approach:** Definitions, business examples, problems and solutions, discussion prompts, and a compact Python lab. Named businesses, customer records, and numerical scenarios are illustrative teaching examples unless a source is explicitly identified.

![Week 11 learning pathway](https://media.geeksforgeeks.org/wp-content/uploads/20250924153111948541/text_processing.webp)

<a id="section-1"></a>

---

## 🧹 **1. Introduction to Text Preprocessing**

### 🔹 **1.1 What is Text Preprocessing?**

**Definition**  
Text preprocessing is the deliberate transformation of raw language into a representation suitable for a particular analytical task. It includes cleaning, normalization, segmentation, and quality checks.

**Example**  
Raw: “Delivery was sooo LATE!!! 😡”  
One controlled view: “delivery was so late ! ! ! angry_emoji”  
A token view can preserve “late” and “angry_emoji”; the original review remains available.

**🚨 Problem**  
The same topic may appear in several surface forms, so counting raw strings fragments the evidence. Aggressive cleaning can also erase emotion and meaning.

**✅ Solution**  
Create derived columns using a documented policy. Keep the raw record unchanged, inspect before-and-after examples, and judge the output against the business objective.

> **Link to Week 10:** Collection answered “Where did this text come from?” Preprocessing answers “Which transformations help this task while preserving relevant evidence?”

### 🔹 **1.2 Noisy Text and Conventional Data Preparation**

**Definition**  
Conventional data preparation checks records, fields, types, missing values, and duplicates. Text preprocessing also operates inside each document on words, punctuation, spelling, and sentence structure.

**Example**  
| Issue | Record-level decision | Text-level decision |
|---|---|---|
| Missing review | Flag or exclude from text analysis | Do not replace with “neutral” |
| Repeated source ID | Investigate repeated collection | Do not count it as a second opinion |
| “GOOD” and “good” | Both are valid strings | Choose whether to normalize case |
| “not helpful” | A valid comment | Preserve the negation |
| Mixed English and Roman Urdu | Retain language metadata | Route to an appropriate language policy |

**🚨 Problem**  
Treating text as an ordinary string column misses important language-specific risks.

**✅ Solution**  
Check the table and the language separately. Validate that every derived text row still maps to its original record.

### 🔹 **1.3 Designing the Preprocessing Pipeline**

**Definition**  
A pipeline is an ordered set of reproducible transformations. Order matters because an early change may remove evidence needed by a later step.

**Example**  
| Stage | Purpose | Why this position matters |
|---|---|---|
| Preserve and audit records | Keep evidence and inspect coverage | Exclusions must remain traceable |
| Parse HTML and normalize Unicode | Recover consistent visible text | Avoid counting tags as words |
| Handle links, emails, contractions, and emoji | Preserve useful signals or replace sensitive spans | Do this before broad punctuation filtering |
| Segment sentences and tokenize | Define analytical units | Keep sentence and document boundaries |
| Create task-specific token views | Select stop words or word normalization | Do not force one view on every task |
| Validate and export | Confirm fidelity and reproducibility | Check the final corpus before modeling |

**🚨 Problem**  
Deleting apostrophes before expanding “isn’t” can produce “isnt,” obscuring its negative meaning.

**✅ Solution**  
Specify an order, check edge cases, and rerun the same function for all records. Keep sentence-aware tokens for phrase analysis and a separate filtered view for optional word-frequency analysis.

### 🔹 **1.4 Choose Transformations for the Task**

**Definition**  
Useful preprocessing depends on the model, language, and question. A transformation that helps a word-count model may damage an entity extractor or a pretrained language model.

**Example**  
| Task | Useful choices | Information to preserve |
|---|---|---|
| Topic exploration with word counts | Consistent case and selected stop words | Domain nouns and phrases |
| Sentiment analysis | Resolve contractions and preserve context | Negation, contrast, intensifiers, emotion |
| Entity extraction | Parse layout and tokenize carefully | Names, case, IDs, amounts, dates |
| Pretrained transformer or LLM input | Follow the model's own preprocessing | Natural wording, punctuation, token order |
| Mixed-language analysis | Language-aware segmentation and normalization | Original script and code-switching |

**🚨 Problem**  
A universal “remove everything except letters” function can destroy the amount, product version, and sentiment in a complaint.

**✅ Solution**  
Write down what the task must retain. Compare the original text, a minimally normalized view, and any additional task-specific view before adopting a policy.

### 🔹 **1.5 Define the Policy Before Writing Code**

**Definition**  
A preprocessing policy records the intended handling of content, uncertain cases, and exclusions.

**Example**  
For this lab, keep raw text, preserve “not/no/never,” retain numbers, map a few known emojis, replace links and emails with markers, and apply English lemmas only to English records.

**🚨 Problem**  
Two analysts can produce different results from the same reviews if their cleaning choices are undocumented.

**✅ Solution**  
Version the policy, record package versions, keep an exclusion audit, and test a small set of meaning-critical examples. The policy is limited to the teaching sample; it is not a complete multilingual or privacy solution.

<a id="section-2"></a>

---

## 🔤 **2. Text Cleaning and Normalization**

### 🔹 **2.1 Cleaning vs Normalization**

**Definition**  
Cleaning removes or handles material that is irrelevant to the task. Normalization maps different surface forms to a consistent form.

**Example**  
Removing an HTML tag is cleaning. Mapping “GOOD” to “good,” “isn’t” to “is not,” or a full-width character to its standard equivalent is normalization.

**🚨 Problem**  
Calling all characters outside A–Z “noise” incorrectly discards valid language and business information.

**✅ Solution**  
Decide whether a span is irrelevant, meaningful, or sensitive. Delete only irrelevant material, preserve or transform meaningful content, and use an appropriate privacy process for sensitive text.

### 🔹 **2.2 Lowercasing and Case Folding**

**Definition**  
Lowercasing standardizes letter case. Case folding is a stronger Unicode operation intended for caseless matching.

**Example**  
“DELIVERY,” “Delivery,” and “delivery” can become “delivery.” However, “US” and “us” may have different meanings, and all capitals can indicate emphasis.

**🚨 Problem**  
Blind lowercasing can erase named-entity distinctions or emotion.

**✅ Solution**  
For a simple English phrase-count view, lowercasing is often useful. Retain the original text and consider an emphasis feature if the task needs it. Follow the model's specified casing policy for model input.

### 🔹 **2.3 Unicode Normalization**

**Definition**  
Unicode normalization gives consistent representations to certain equivalent character sequences. NFC combines canonical equivalents; NFKC additionally normalizes compatibility forms.

**Example**  
NFKC can convert full-width “ＰＫＲ” to “PKR.” A composed “é” and an “e” plus a combining accent can be made consistent without deleting the accent.

**🚨 Problem**  
Visually similar text can compare unequally, while indiscriminate accent or script deletion can change meaning.

**✅ Solution**  
Choose and document a normalization form. The lab uses NFKC for its selected English-oriented fixtures and keeps raw text. Compatibility normalization is not automatically suitable for every script, identifier, or archival task.

**Important distinction:** Unicode normalization does not translate Urdu or Roman Urdu into English.

### 🔹 **2.4 Removing HTML and Web Artifacts**

**Definition**  
HTML parsing distinguishes page markup from visible text. Tags, scripts, styles, menus, and boilerplate need different handling.

**Example**  
`<p>Fast <b>delivery</b> &amp; good support.</p>` should yield “Fast delivery & good support.” A script's contents should not become customer feedback.

**🚨 Problem**  
A broad regular expression can break on nested markup or join adjacent words. Scraping an entire page can include navigation and advertisements.

**✅ Solution**  
Extract the review field first. Parse markup with an HTML parser, remove non-content elements, and join text segments with spaces. Inspect malformed examples instead of assuming every text value is a full webpage.

### 🔹 **2.5 URLs, Emails, and Meaningful Markers**

**Definition**  
URL and email handling removes or replaces link and contact spans. A marker preserves the fact that a span occurred without retaining its full value in the analytical view.

**Example**  
“Refund not received; email learner@example.invalid” becomes “refund not received; email EMAIL_TOKEN.” A web link can become URL_TOKEN.

**🚨 Problem**  
Deleting links can join words, leak identifiers through fragments, or lose information about customers requesting contact.

**✅ Solution**  
Replace matched spans with space-delimited markers and validate examples. Keep URL domains only if relevant and permitted. Regex replacement alone is not reliable anonymization: names, phone numbers, addresses, and contextual identifiers can remain.

> All contact strings in this notebook are synthetic. A real privacy workflow requires broader detection, review, access controls, and retention rules.

### 🔹 **2.6 Punctuation, Special Characters, and Whitespace**

**Definition**  
Punctuation helps express structure and tone. Whitespace normalization standardizes repeated spaces, tabs, and line breaks.

**Example**  
“not-good” can be tokenized as “not” and “good.” Deleting the hyphen without adding a boundary produces the invalid merged form “notgood.” “C++,” “5%,” and “A-10” may be meaningful domain terms.

**🚨 Problem**  
Removing punctuation too early destroys sentence boundaries, contractions, currency expressions, and emoticons.

**✅ Solution**  
Handle protected patterns first, then define tokens for the task. Preserve sentence boundaries for phrase counting; collapse repeated whitespace only after meaningful boundaries have been handled. Keep raw text for punctuation-sensitive tasks.

### 🔹 **2.7 Numbers, Amounts, and Product Identifiers**

**Definition**  
Numbers may describe prices, delays, quantities, model versions, or private identifiers; these roles require different treatment.

**Example**  
“PKR 2,500 was deducted twice” contains an amount and a repeat-event signal. “Battery lasts 2 hours” differs materially from “Battery lasts 20 hours.”

**🚨 Problem**  
Removing all digits prevents analysts from comparing severity or identifying a product version.

**✅ Solution**  
Retain meaningful quantities and units; normalize thousands separators only when the format is unambiguous. The lab keeps numbers and treats its English-formatted “2,500” as “2500.” Do not apply that assumption to locales where the comma is a decimal separator.

### 🔹 **2.8 Emojis and Emoticons**

**Definition**  
Emojis and emoticons can communicate emotion, sarcasm, or emphasis. Their interpretation depends on context and domain.

**Example**  
“Excellent support 😊” and “Great, the app crashed again 🙄” need different interpretations. “:)” may be useful in a sentiment task.

**🚨 Problem**  
Removing every emoji can lose sentiment information; assigning one fixed sentiment to every emoji can also be wrong.

**✅ Solution**  
Keep them, map a reviewed subset to descriptive tokens, or retain a separate feature. The lab maps only 😊, 😡, and 🙄. These markers describe the symbol; they are not validated sentiment labels.

### 🔹 **2.9 Contractions, Slang, and Abbreviations**

**Definition**  
Contraction expansion restores shortened grammatical forms. Slang and abbreviation normalization maps known informal variants using a task-specific dictionary.

**Example**  
“can't” becomes “can not”; “isn’t” becomes “is not.” A reviewed shorthand such as “plz” may become “please.” But “US” and “us,” or a product code and a slang term, should not be merged casually.

**🚨 Problem**  
A general replacement inside larger words can create errors; unfamiliar slang may be ambiguous.

**✅ Solution**  
Normalize apostrophes, apply whole-token or boundary-aware replacements, and maintain a small auditable dictionary. Preserve unknown forms for review. The lab's contraction dictionary is intentionally limited to the examples used.

### 🔹 **2.10 Spelling Errors and Repeated Characters**

**Definition**  
Spelling correction and repeated-character normalization attempt to reduce superficial variation. Neither is guaranteed to preserve meaning.

**Example**  
“sooo late” can map to “so late” for a phrase-count view. But shortening every doubled letter would damage “book,” “cool,” and brand names.

**🚨 Problem**  
Automatic correction can replace a valid name, technical term, or Roman Urdu word with an unrelated English word.

**✅ Solution**  
Use validated, domain-specific mappings or an evaluated correction method. The lab normalizes only the known variant “sooo.” Retain raw emphasis and review uncertain words instead of inventing a correction.

### 🔹 **2.11 Multilingual and Mixed-Language Text**

**Definition**  
Multilingual preprocessing respects script, language-specific morphology, and language changes inside a document.

**Example**  
“Product acha hai but delivery bohat slow thi” mixes English and Roman Urdu. Urdu-script feedback is also valid business text.

**🚨 Problem**  
An English stop-word list will not handle “hai” or “thi.” A filter limited to `[a-z]` can delete non-Latin scripts entirely.

**✅ Solution**  
Preserve Unicode and language metadata. Use language-appropriate tools or route mixed-language records for a separate workflow. Do not run an English lemmatizer across all records and report the output as multilingual analysis.

### 🔹 **2.12 A Worked Before-and-After Policy**

**Definition**  
A transformation audit records what changed and why, rather than judging cleaning by how short the result became.

**Example**  
Raw: `<p>I can't get a refund!!! 😡 Visit https://example.com/help</p>`

| Stage | Illustrative result |
|---|---|
| Parse HTML | I can't get a refund!!! 😡 Visit https://example.com/help |
| Replace URL and normalize contraction | I can not get a refund!!! 😡 Visit URL_TOKEN |
| Normalize case and map emoji | i can not get a refund!!! angry_emoji visit url_token |
| Word-token view | i, can, not, get, a, refund, angry_emoji, visit, url_token |
| Optional selected-stop-word view | can, not, get, refund, angry_emoji, visit, url_token |

**🚨 Problem**  
A polished-looking result such as “refund” loses the failure, negation, and emotion.

**✅ Solution**  
Verify preserved meaning using the original sentence. Keep several views when tasks need different levels of normalization. A stop-word policy is a choice, not a universal truth.

<a id="section-3"></a>

---

## ✂️ **3. Tokenization**

### 🔹 **3.1 What is a Token?**

**Definition**  
A token is a unit produced by a tokenizer. Depending on the method, it may be a word, punctuation mark, subword, number, or symbol.

**Example**  
A word tokenizer may split “Refund not received.” into “Refund,” “not,” “received,” and “.” A filtered word view may omit the final punctuation token.

**🚨 Problem**  
Assuming one token always equals one word leads to incorrect counts and misleading comparisons across tokenizers.

**✅ Solution**  
State the tokenizer and what its output represents. Distinguish character count, whitespace word count, token count, and vocabulary size.

### 🔹 **3.2 Sentence Tokenization**

**Definition**  
Sentence tokenization segments a document into sentences or sentence-like units.

**Example**  
“Support replied. Refund is pending.” contains two sentences. The boundary matters because “replied refund” is not an original adjacent phrase.

**🚨 Problem**  
Splitting on every period can break decimals and abbreviations: “PKR 4.50” and “Dr. Ali” require care.

**✅ Solution**  
Use a language-appropriate sentence tokenizer for real data. The lab uses a deliberately simple punctuation-plus-space rule for the controlled sample and explicitly tests its scope. It is not a general sentence detector.

### 🔹 **3.3 Word Tokenization and Whitespace Splitting**

**Definition**  
Whitespace splitting uses spaces to separate units. Rule-based tokenization can treat punctuation, contractions, hashtags, and numbers more carefully.

**Example**  
| Input | Whitespace split | Possible rule-based tokens |
|---|---|---|
| “late!!!” | late!!! | late, !, !, ! |
| “not-good” | not-good | not, good |
| “can't pay” | can't, pay | depends on contraction policy |
| “PKR 2500” | PKR, 2500 | PKR, 2500 |

**🚨 Problem**  
Whitespace-only tokens often retain punctuation variants; a rule that fits one domain may mishandle another.

**✅ Solution**  
Choose a tokenizer, document how it handles edge cases, and inspect actual outputs. NLTK provides word-oriented tokenizers; the lab uses resource-free NLTK examples and a documented corpus regex.

### 🔹 **3.4 Rule-Based Tokenization in the Lab**

**Definition**  
A regular-expression tokenizer extracts character sequences that match an explicit pattern.

**Example**  
The lab keeps Unicode word sequences and numeric forms. Markers such as “angry_emoji” remain one token; “Wi-Fi” becomes “wi” and “fi” in the frequency view.

**🚨 Problem**  
A concise regex is easy to audit but cannot solve all morphology, emoji, or mixed-script segmentation problems.

**✅ Solution**  
Use it only within the stated scope. Keep the original text, protect meaningful markers first, and add targeted examples before expanding the data source.

**Compare:** Is splitting “Wi-Fi” acceptable for your chosen business task? What would change for product-code extraction?

### 🔹 **3.5 Subword Tokenization**

**Definition**  
Subword tokenizers split text into learned pieces, allowing a model to represent rare or unfamiliar words using smaller units.

**Example**  
A tokenizer might represent “unhelpful” using pieces such as “un” and “helpful.” This is illustrative; the actual pieces depend on the tokenizer vocabulary and algorithm.

**🚨 Problem**  
Hand-created word tokens do not match the token IDs expected by a pretrained transformer. Token counts can differ by model and language.

**✅ Solution**  
For a pretrained model, use its matching tokenizer and preserve the input expected by that model. Do not manually stem and reassemble text unless the model's evaluated workflow requires it.

### 🔹 **3.6 Tokenization Changes the Questions You Can Ask**

**Definition**  
Tokenization defines the units counted and the boundaries used to form phrases.

**Example**  
Word tokens help count “refund.” Sentence-aware word tokens help distinguish “refund pending” from words separated by a sentence break.

**🚨 Problem**  
Flattening all reviews into one long token list creates phrases that no customer wrote.

**✅ Solution**  
Maintain the hierarchy: corpus, document, sentence, token. Generate phrases within a sentence or a clearly documented document policy, then aggregate counts across records.

<a id="section-4"></a>

---

## 🛑 **4. Stop-Word Removal**

### 🔹 **4.1 What Are Stop Words?**

**Definition**  
Stop words are words selected for removal in a particular text-processing task, often because frequent function words add little to that task's word-count representation.

**Example**  
“the,” “a,” “an,” “is,” and “of” are common candidates. There is no universally correct stop-word list for all languages or tasks.

**🚨 Problem**  
Frequent words can dominate a simple frequency table, hiding product or service terms.

**✅ Solution**  
Evaluate a documented list against the objective. Keep a minimally processed view and create a separate filtered view if the analysis benefits from it.

### 🔹 **4.2 Negation, Contrast, and Intensifiers**

**Definition**  
Negation changes polarity; contrast links competing opinions; intensifiers modify strength.

**Example**  
“not good,” “never arrived,” “no refund,” “good but slow,” and “very poor” contain small words that can materially change interpretation.

**🚨 Problem**  
Removing “not” turns “not good” into “good.” Removing “but” hides the relation between positive product quality and negative delivery.

**✅ Solution**  
Protect task-relevant terms such as “not,” “no,” “never,” and often “but” or “very.” Expanding contractions must happen before applying the list. Verify that the protected terms survive.

### 🔹 **4.3 Domain-Specific Stop Words**

**Definition**  
Domain stop words are frequent terms considered uninformative for a narrowly defined analysis, such as a repeated brand name in a single-brand dataset.

**Example**  
A brand name might add little to one retailer's topic list, but it is essential in competitor analysis. “bank” may be redundant in one dataset and a key entity in another.

**🚨 Problem**  
Removing “delivery” because it is frequent would eliminate the very topic an operations team wants to study.

**✅ Solution**  
Justify each domain exclusion using the question and sample evidence. A word's frequency alone is not a sufficient reason to remove it.

### 🔹 **4.4 When to Retain Stop Words**

**Definition**  
Stop-word removal is optional and often unsuitable for tasks that rely on natural word order or grammatical context.

**Example**  
Question answering, sentence generation, transformer input, and phrase analysis commonly need function words. “out of stock” loses its original form if “of” is removed.

**🚨 Problem**  
An aggressively filtered corpus may no longer be readable or compatible with a chosen model.

**✅ Solution**  
Use full normalized tokens for phrase analysis and a separate filtered token list for optional count exploration. In the lab, n-grams are formed before stop-word removal.

### 🔹 **4.5 Manually Audit a Stop-Word Policy**

**Definition**  
A policy audit checks whether removals affect the interpretation required by the task.

**Example**  
Start with “The service is not very good, but the refund arrived.” Identify the service opinion and refund event before proposing removals.

**🚨 Problem**  
A generic list may remove more than the analyst expects.

**✅ Solution**  
Underline meaning-bearing terms, compare the filtered output with the source, and explain each removal. Different tasks can justify different lists; unsupported loss of meaning is the error.

**Suggested audit:** Protect “not,” consider retaining “very” and “but,” and retain “refund.” Do not infer that “arrived” means the entire service experience was positive.

<a id="section-5"></a>

---

## 🌱 **5. Stemming and Lemmatization**

### 🔹 **5.1 Word Normalization**

**Definition**  
Word normalization groups related surface forms so they can be analyzed together. Stemming and lemmatization use different mechanisms and produce different outputs.

**Example**  
“payments” and “payment” can be grouped. But “payment,” “refund,” and “transfer” are related concepts, not merely inflected forms of the same word.

**🚨 Problem**  
Counting every inflected form separately may fragment a topic, while merging different concepts destroys useful distinctions.

**✅ Solution**  
Choose word normalization separately from semantic grouping. Preserve unnormalized tokens for comparison and avoid assuming that a common stem implies the same meaning.

### 🔹 **5.2 Stemming and the Porter Stemmer**

**Definition**  
Stemming applies rules to remove or alter affixes. The resulting stem need not be a dictionary word. Porter is an English stemming algorithm implemented in NLTK.

**Example**  
With the tested NLTK implementation, “deliveries” becomes “deliveri,” “delivered” becomes “deliv,” and “running” becomes “run.”

**🚨 Problem**  
Stems can be hard for managers to read, and related-looking words may be merged incorrectly or remain separate.

**✅ Solution**  
Stemming may suit a compact search or count representation when evaluated for the task. Use original phrases or readable labels in reports rather than displaying stems as if they were natural language.

### 🔹 **5.3 Snowball Stemming**

**Definition**  
Snowball refers to a family of language-specific stemming algorithms and their implementation framework. NLTK exposes a SnowballStemmer with an explicit language choice.

**Example**  
For the tested English stemmer, “payments” becomes “payment.” Porter and Snowball may agree on some words and differ on others.

**🚨 Problem**  
An English stemmer applied to Urdu or Roman Urdu does not become a multilingual method.

**✅ Solution**  
Select the supported language deliberately, compare example outputs, and leave incompatible-language records for a different workflow. Do not assume every stemmer version or mode behaves identically.

### 🔹 **5.4 Lemmatization and Dictionary Forms**

**Definition**  
Lemmatization maps a word form to a lemma, commonly its dictionary form, using lexical information, rules, and sometimes grammatical context.

**Example**  
“children” can become “child,” “running” can become “run,” and “deliveries” can become “delivery.” “better” may map to “good” under an adjective-oriented interpretation.

**🚨 Problem**  
The same spelling can play different grammatical roles: “saw” may refer to a tool or the past tense of “see.” A context-free lookup can choose the wrong interpretation.

**✅ Solution**  
Use an approach appropriate to the task. For more demanding work, include part-of-speech and context. The lab uses Simplemma's packaged English dictionary, so it runs without a separate corpus download, but it is not a context-aware grammatical analyzer.

### 🔹 **5.5 Part of Speech and NLTK WordNet**

**Definition**  
Part of speech describes a word's grammatical role, such as noun, verb, adjective, or adverb. NLTK's WordNetLemmatizer accepts a part-of-speech argument and defaults to noun.

**Example**  
`lemmatize("running", pos="v")` can return “run”; using the noun default may leave “running” unchanged. The WordNet corpus must be installed before using that lemmatizer.

**🚨 Problem**  
A tool may run successfully while using an unsuitable default for many tokens.

**✅ Solution**  
Check the tool's resource requirements and grammatical assumptions. The executed lab compares NLTK stemmers with Simplemma lemmas and states this distinction explicitly; it does not present dictionary lookup as automatic part-of-speech tagging.

### 🔹 **5.6 Stemming vs Lemmatization**

**Definition**  
The practical choice depends on representation quality, readability, language support, speed, and downstream evaluation.

**Example**  
| Dimension | Stemming | Lemmatization |
|---|---|---|
| Main mechanism | Affix rules | Lexicon/rules, sometimes grammatical context |
| Typical output | May be a non-word | Intended dictionary form |
| “children” in the tested examples | children | child |
| “deliveries” in the tested examples | deliveri | delivery |
| Key limitation | Over- or under-stemming | Lexicon gaps and ambiguous context |
| Evaluation question | Does grouping help retrieval or prediction? | Does the lemma preserve the intended meaning? |

**🚨 Problem**  
Students may assume lemmatization is always correct or that applying both methods in sequence is automatically better.

**✅ Solution**  
Compare alternative token views. Do not stem a token and then lemmatize the invented stem by default. Keep original phrases for interpretation and evaluate any transformation against the business task.

<a id="section-6"></a>

---

## 🔍 **6. N-Grams and Text Pattern Discovery**

### 🔹 **6.1 What is an N-Gram?**

**Definition**  
A token n-gram is a contiguous sequence of n tokens within a defined unit. A unigram contains one token, a bigram two, and a trigram three.

**Example**  
Sentence tokens: **delivery / was / not / late**

| Type | Sequences |
|---|---|
| Unigrams | delivery; was; not; late |
| Bigrams | delivery was; was not; not late |
| Trigrams | delivery was not; was not late |

**🚨 Problem**  
An unordered bag of individual words loses the “not late” relationship.

**✅ Solution**  
Use n-grams when local order helps, while recognizing that short phrases still do not capture all sentence meaning.

### 🔹 **6.2 Counting N-Grams Correctly**

**Definition**  
For one token sequence of length m, the number of contiguous n-grams is max(m − n + 1, 0). This is a count of positions, not necessarily distinct phrases.

**Example**  
The four-token sequence “delivery was not late” has four unigrams, three bigrams, and two trigrams. Repeated phrases can occupy more than one position.

**🚨 Problem**  
Confusing total occurrences with unique phrases produces inconsistent tables.

**✅ Solution**  
State whether you count occurrences, distinct n-grams, or documents containing a phrase. Compute sequences within each sentence and aggregate only after generating them.

$$N_{\text{positions}} = \max(m-n+1,\ 0)$$

### 🔹 **6.3 Do Not Create Phrases Across Boundaries**

**Definition**  
An n-gram boundary policy defines where adjacency is allowed. This lab uses contiguous tokens within each sentence and within each review.

**Example**  
Document A: “refund pending.” Document B: “delivery late.” Concatenating them creates “pending delivery,” although no customer wrote that phrase.

**🚨 Problem**  
Removing stop words or joining documents before forming n-grams invents adjacency. “delivery was not late” must not become evidence for the original phrase “delivery late.”

**✅ Solution**  
Build n-grams from full normalized sentence tokens. Keep stop-word-filtered tokens for a separate purpose. Never join all reviews before phrase generation.

### 🔹 **6.4 Frequency and Document Frequency**

**Definition**  
Occurrence frequency counts every appearance of a phrase. Document frequency counts how many different documents contain it at least once.

**Example**  
If one review repeats “refund pending” three times, its occurrence count is three but its document frequency is one.

**🚨 Problem**  
A repetitive reviewer can dominate raw counts; duplicate collection can amplify the distortion.

**✅ Solution**  
Remove repeated collection according to source identity, report both measures when useful, and keep the eligible-document denominator visible. In a multi-topic analysis, phrase shares can overlap.

### 🔹 **6.5 Business Uses and Limits of Phrase Counts**

**Definition**  
Phrase counts support exploratory discovery of recurring expressions; they are not automatic sentiment labels, severity measures, or causal explanations.

**Example**  
“not late” contains “late” but expresses a different experience from “very late.” “great service” may be literal or sarcastic depending on context.

**🚨 Problem**  
A high-frequency phrase can be positive, negative, neutral, boilerplate, or an artifact of the preprocessing policy.

**✅ Solution**  
Read supporting reviews, compare phrases in context, and validate any business labels. Use the phrase table to decide what to investigate, then connect validated issues to operational evidence.

### 🔹 **6.6 Unigrams, Bigrams, or Trigrams?**

**Definition**  
Increasing n adds local context but creates more possible features and fewer repeated observations for each feature.

**Example**  
Unigrams can reveal “refund”; bigrams can reveal “not received”; trigrams can reveal “refund not received.” A longer exact phrase may miss paraphrases.

**🚨 Problem**  
Long phrases become sparse; short phrases may be ambiguous. Small samples do not support stable business rankings.

**✅ Solution**  
Compare several n values, require interpretable evidence, and avoid presenting the top phrase as a universal customer priority. Phrase features and model representations can be developed further in later topics.

<a id="section-7"></a>

---

## 📚 **7. Preparing a Text Corpus**

### 🔹 **7.1 Corpus, Document, Token, and Vocabulary**

**Definition**  
A corpus is an organized collection of documents. A document is one analytical text unit. Tokens are units within documents, and the vocabulary is the set of distinct tokens under a specified preprocessing policy.

**Example**  
If each review is a document, 12 eligible reviews form a 12-document English corpus. Ten appearances of “refund” contribute ten token occurrences but one vocabulary entry.

**🚨 Problem**  
“Corpus size” is ambiguous unless it identifies documents, tokens, characters, or bytes.

**✅ Solution**  
Report document count, token count, and vocabulary size separately. State the unit of analysis and policy used to produce them.

### 🔹 **7.2 Preserve Raw Text and Derived Views**

**Definition**  
A traceable corpus links each processed representation to its original record and processing policy.

**Example**  
| Field | Meaning |
|---|---|
| ingestion_id | A collected row, including repeated collection |
| review_key | Original source plus source record ID |
| raw_text | Original collected content |
| normalized_text | Text after the documented cleaning policy |
| sentence_tokens | Tokens grouped by sentence |
| filtered_tokens | Optional selected-stop-word view |
| lemmas | English dictionary-normalized view where eligible |
| status and policy_version | Inclusion decision and transformation history |

**🚨 Problem**  
Overwriting raw text prevents an analyst from explaining an unexpected result or correcting the pipeline later.

**✅ Solution**  
Keep the original and derived columns together or joined by stable IDs. Preserve exclusions in an audit file, not just the final accepted corpus.

### 🔹 **7.3 Missing, Blank, Duplicate, and Conflicting Records**

**Definition**  
Missing text, whitespace-only text, repeated source records, and conflicting copies are distinct data-quality conditions.

**Example**  
A null comment has no text. “   ” contains no usable words. A repeated source ID may be a collection duplicate. The same source ID with different wording may be an edited review or a collection error.

**🚨 Problem**  
Replacing missing values with the string “nan” adds a fake token. Removing every repeated text string can delete separate customers who legitimately wrote the same comment.

**✅ Solution**  
Flag missing and blank text; deduplicate repeated source identity under a stated policy; inspect conflicts instead of silently choosing a copy. Text similarity can flag suspected duplicates, but it is not proof of shared identity.

### 🔹 **7.4 Reusable Functions and Idempotence**

**Definition**  
A reusable function applies the same policy to every record. An idempotent cleaning function produces the same result when applied again to its own cleaned output.

**Example**  
Collapsing repeated whitespace should not keep changing a document on every run. A replacement rule should not expand its own replacement repeatedly.

**🚨 Problem**  
Manual edits and uncontrolled replacement order make results difficult to reproduce.

**✅ Solution**  
Implement one named function, keep policy constants visible, and test representative inputs. Idempotence is useful for the normalized-text stage, but does not establish that the meaning is correct.

### 🔹 **7.5 Corpus Quality Checks**

**Definition**  
Quality checks verify record alignment, exclusions, language coverage, token validity, and preserved meaning.

**Example**  
| Check | What it can catch |
|---|---|
| Raw versus processed sample review | Lost negation or distorted wording |
| Unique review keys in final corpus | Repeated collection |
| No empty token lists | Documents emptied by cleaning |
| Protected-token checks | Accidental removal of “not” |
| Exclusion counts by reason | Hidden loss of languages or records |
| Known number and emoji examples | Over-aggressive symbol removal |
| Export round trip | Lost Unicode or broken token-list serialization |

**🚨 Problem**  
A pipeline can execute without errors while producing an unsuitable corpus.

**✅ Solution**  
Combine programmatic checks with reading real before-and-after examples. Report coverage limitations alongside counts; passing checks does not turn synthetic data into business evidence.

### 🔹 **7.6 Preparing for Subsequent Modeling**

**Definition**  
A corpus handoff should include the unit of analysis, provenance, text policy, labels if available, and a plan for separating training from evaluation data.

**Example**  
A later supervised model may use review text to predict a verified category. A vocabulary or TF-IDF representation learned from all documents before splitting can leak evaluation information.

**🚨 Problem**  
Near-duplicate reviews can cross splits, and fitted preprocessing can inadvertently learn from held-out records.

**✅ Solution**  
Group related or duplicate records appropriately, split before fitting data-dependent transformations, and preserve a genuinely held-out set. These notebooks stop at corpus preparation; they do not train or evaluate a predictive model.

<a id="section-8"></a>

---

## 💼 **8. Business Case Studies and Error-Spotting Activities**

### 🔹 **8.1 E-Commerce: Preserve the Complaint**

**Definition**  
A useful product-review corpus retains the problem, the affected aspect, and any important quantity.

**Example**  
“Battery lasts only 2 hours. Not worth PKR 5,000.”

**🚨 Problem**  
A pipeline removes numbers and “not,” leaving “battery lasts hours worth pkr.” The dissatisfaction and numerical context are damaged.

**✅ Solution**  
Retain the amount and duration, preserve negation, and keep sentence boundaries. Do not infer a technical battery fault solely from this subjective review.

**Task:** Propose separate features for the product aspect, quantity, and opinion. Which claims would require a product test?

### 🔹 **8.2 Banking: Keep Transaction Evidence Safe and Useful**

**Definition**  
Banking complaints often contain a mix of service descriptions, amounts, dates, and identifying information.

**Example**  
“Payment failed twice! PKR 2,500 was deducted.”

**🚨 Problem**  
Deleting all digits removes the amount; publishing complete customer complaints could expose private information.

**✅ Solution**  
Retain relevant quantities in an authorized analytical environment, minimize identifying fields, and validate patterns with appropriate operational records. In this notebook, the text is synthetic and requires no real account access.

**Task:** Separate an amount from an account identifier. Explain why one broad digit-removal rule is inadequate.

### 🔹 **8.3 Hospitality: Avoid Invented Phrases**

**Definition**  
Hotel feedback often contains multiple service aspects across several sentences.

**Example**  
“Room was clean. Breakfast was not good.”

**🚨 Problem**  
Joining tokens across sentences creates “clean breakfast”; removing negation suggests a positive breakfast opinion.

**✅ Solution**  
Retain sentence boundaries and “not.” Review the room and breakfast aspects separately. Generate phrase candidates within each sentence.

**Task:** List the valid bigrams and name two invalid phrases that an incorrect pipeline could create.

### 🔹 **8.4 Retail: Domain Terms and Shared Wording**

**Definition**  
Retail language includes product codes, brand names, stock status, and ordinary phrases repeated by different customers.

**Example**  
Two independent customers write “out of stock,” while a page is accidentally downloaded twice.

**🚨 Problem**  
Deleting “of” destroys the original phrase, and deduplicating only by text erases a valid second customer report.

**✅ Solution**  
Preserve full tokens for phrase analysis and deduplicate repeated collection using source IDs. Keep independently authored matching text unless evidence supports a different policy.

**Task:** Explain the difference between duplicate collection and repeated customer experience.

### 🔹 **8.5 Find and Fix the Preprocessing Mistakes**

For each row, identify the lost information, propose a repair, and state which task is affected.

| Original | Faulty output |
|---|---|
| The service is not good. | service good |
| Product acha hai but delivery slow thi. | product delivery slow |
| Refund pending. Delivery late. | pending delivery |
| PKR 2,500 deducted twice. | pkr deducted twice |
| `<b>slow</b><i>delivery</i>` | slowdelivery |
| Missing value | nan |

<details>
<summary><strong>Reveal the repair guide</strong></summary>

- Preserve negation before optional stop-word filtering.
- Retain the mixed-language record and use a suitable language workflow.
- Respect sentence boundaries when generating phrases.
- Retain meaningful amounts under a stated numeric-format policy.
- Parse HTML and join text nodes with spaces.
- Flag missing text instead of converting it to a literal word.

</details>

<a id="section-9"></a>

---

## 📝 **9. Summary and Reinforcement**

### 🔹 **9.1 Compare the Main Techniques**

| Technique | Main purpose | Main risk |
|---|---|---|
| Cleaning | Handle markup and irrelevant spans | Removing useful evidence |
| Normalization | Standardize selected surface forms | Merging distinct meanings |
| Tokenization | Define analytical units | Breaking important expressions |
| Stop-word removal | Reduce selected low-value words | Losing negation and context |
| Stemming | Group forms using affix rules | Unreadable or incorrect stems |
| Lemmatization | Map forms to dictionary entries | Wrong interpretation without context |
| N-grams | Preserve local token order | Sparse features or invented adjacency |
| Corpus validation | Check fidelity and coverage | Assuming technical success proves meaning |

### 🔹 **9.2 Scenario-Based Knowledge Check**

1. Why should raw and normalized text remain separate?
2. What must happen before removing apostrophes from “isn't helpful”?
3. Why is “no refund” different from “refund”?
4. What is the difference between a stem and a lemma?
5. How many bigram positions are in a sentence of five tokens?
6. Why should n-grams be formed before optional stop-word removal?
7. How do phrase occurrences differ from the number of reviews mentioning the phrase?
8. Why should an English lemmatizer not be applied indiscriminately to mixed-language feedback?
9. What is the problem with removing duplicates only by review text?
10. Which preprocessing steps must be fitted on training data only when modeling begins?

<details>
<summary><strong>Reveal concise answers</strong></summary>

1. To audit transformations and recover original evidence.
2. Expand or otherwise preserve its negation.
3. “No” changes the event or opinion being reported.
4. A stem is a rule-based reduced form; a lemma is an intended dictionary form.
5. Four.
6. Filtering can create adjacency that was absent in the source.
7. A phrase can appear repeatedly in one review; document frequency counts that review once.
8. Its dictionary and morphology do not cover every language or code-switching pattern.
9. Independent customers may legitimately use the same wording.
10. Data-dependent steps such as learned vocabularies and TF-IDF weights.

</details>

> **Takeaway:** Good preprocessing preserves the evidence needed by the task. A shorter string is not necessarily a better representation.

<a id="section-10"></a>

---

## 🔗 **10. Additional Reading and Resources**

- [NLTK — Tokenization examples](https://www.nltk.org/howto/tokenize.html) — inspect tokenizer-specific behavior.
- [NLTK — Stemming](https://www.nltk.org/api/nltk.stem.html) — Porter and Snowball implementations.
- [NLTK — WordNet lemmatization](https://www.nltk.org/api/nltk.stem.wordnet.html) — part-of-speech arguments and lexical resources.
- [Simplemma — Official documentation](https://adbar.github.io/simplemma/) — the dictionary-based lemmatizer used in this lab.
- [Python — Regular expressions](https://docs.python.org/3/library/re.html) — patterns and boundary-aware replacement.
- [Python — Unicode normalization](https://docs.python.org/3/library/unicodedata.html) — NFC and NFKC.
- [Pandas — Working with text](https://pandas.pydata.org/docs/user_guide/text.html) — missing-aware string operations.
- [spaCy — Linguistic features](https://spacy.io/usage/linguistic-features) — optional extension for tokenization, grammatical analysis, and lemmatization.

**Scope of the practical work:** English-oriented, interpretable preprocessing on synthetic reviews. Mixed-language records are preserved and reported separately. The notebook does not claim to infer sentiment, translate text, anonymize arbitrary records, or diagnose real business performance.

<a id="lab"></a>

---

## 💻 **LAB WEEK 11: Customer Review Cleaning and Corpus Preparation**

### **Goal**

Load the Week 10 review data, preserve original records, apply a documented cleaning policy, compare token views, count phrases, and export an English corpus with an audit trail.

### **Setup**

Use Python 3.10 or newer in Jupyter or Google Colab. Run the cells in order. If needed, run the following in a new code cell:

```python
%pip install pandas==2.2.3 beautifulsoup4==4.15.0 nltk==3.10.3 simplemma==2.0.0
```

The NLTK tokenizers and stemmers used below need **no separate NLTK corpus download**. Simplemma includes the lexical data needed for this English demonstration. Package installation needs internet; the default lab does not need it afterwards.

**Input:** `ba3002_text_mining_data/week10_customer_reviews.csv`, or the same embedded teaching records if that file is absent.

**Output:** a complete processing audit, a filtered English corpus, phrase counts, and a policy record. List-valued columns use JSON inside CSV so they can be read back correctly.

In [ ]:
import json  # Load and save structured records and token lists.
import re  # Apply explicit, inspectable text patterns.
import unicodedata  # Normalize selected Unicode forms.
from collections import Counter  # Count token and phrase occurrences.
from pathlib import Path  # Work with portable file paths.
from importlib.metadata import version  # Record installed package versions.
import pandas as pd  # Organize the review corpus and audit tables.
from bs4 import BeautifulSoup  # Extract readable text from markup.
from nltk.tokenize import TreebankWordTokenizer, TweetTokenizer  # Demonstrate tokenizers without downloaded corpora.
from nltk.stem import PorterStemmer, SnowballStemmer  # Compare English stemming algorithms.
from simplemma import lemmatize  # Use packaged English lexical data for lemmas.
from IPython.display import display  # Render compact output tables.
pd.set_option("display.max_colwidth", 90)  # Bound long preview strings.
output_dir = Path("ba3002_text_mining_data")  # Share the Week 10 handoff folder.
output_dir.mkdir(exist_ok=True)  # Create it when this notebook runs independently.
policy_version = "ba3002_week11_v1"  # Identify the applied cleaning policy.

### **Step 1 — Load the Review Data**

The embedded fallback makes this notebook independently runnable. If the Week 10 CSV exists, it is used as the authoritative input. The language labels in this synthetic dataset are supplied for teaching; the code does not automatically detect language.

In [ ]:
# Embedded fallback is identical to the Week 10 exported teaching dataset.
embedded_records = [{'ingestion_id': 'I001',
  'source': 'web',
  'source_record_id': 'W01',
  'business_area': 'e-commerce',
  'language': 'en',
  'rating': 3,
  'raw_text': 'Delivery was late, but the product is good.',
  'review_key': 'web:W01'},
 {'ingestion_id': 'I002',
  'source': 'web',
  'source_record_id': 'W02',
  'business_area': 'e-commerce',
  'language': 'en',
  'rating': 5,
  'raw_text': '<p>Excellent support! 😊</p>',
  'review_key': 'web:W02'},
 {'ingestion_id': 'I003',
  'source': 'web',
  'source_record_id': 'W03',
  'business_area': 'banking',
  'language': 'en',
  'rating': 1,
  'raw_text': "I can't get a refund. Support isn't helpful.",
  'review_key': 'web:W03'},
 {'ingestion_id': 'I004',
  'source': 'web',
  'source_record_id': 'W04',
  'business_area': 'hospitality',
  'language': 'en',
  'rating': 3,
  'raw_text': 'Room was clean, but Wi-Fi was slow.',
  'review_key': 'web:W04'},
 {'ingestion_id': 'I005',
  'source': 'web',
  'source_record_id': 'W05',
  'business_area': 'retail',
  'language': 'en',
  'rating': 3,
  'raw_text': 'Checkout queue was long; staff were helpful.',
  'review_key': 'web:W05'},
 {'ingestion_id': 'I006',
  'source': 'web',
  'source_record_id': 'W06',
  'business_area': 'e-commerce',
  'language': 'en',
  'rating': 5,
  'raw_text': 'Delivery was not late. Great service!',
  'review_key': 'web:W06'},
 {'ingestion_id': 'I007',
  'source': 'web',
  'source_record_id': 'W01',
  'business_area': 'e-commerce',
  'language': 'en',
  'rating': 3,
  'raw_text': 'Delivery was late, but the product is good.',
  'review_key': 'web:W01'},
 {'ingestion_id': 'I008',
  'source': 'api',
  'source_record_id': 'A01',
  'business_area': 'banking',
  'language': 'en',
  'rating': 1,
  'raw_text': 'Payment failed twice! PKR 2,500 was deducted.',
  'review_key': 'api:A01'},
 {'ingestion_id': 'I009',
  'source': 'api',
  'source_record_id': 'A02',
  'business_area': 'e-commerce',
  'language': 'en',
  'rating': 2,
  'raw_text': 'Delivery was sooo LATE!!! 😡 See https://example.com/order/42',
  'review_key': 'api:A02'},
 {'ingestion_id': 'I010',
  'source': 'api',
  'source_record_id': 'A03',
  'business_area': 'retail',
  'language': 'en',
  'rating': 1,
  'raw_text': 'Battery lasts only 2 hours. Not worth PKR 5,000.',
  'review_key': 'api:A03'},
 {'ingestion_id': 'I011',
  'source': 'api',
  'source_record_id': 'A04',
  'business_area': 'banking',
  'language': 'en',
  'rating': 1,
  'raw_text': 'Great, the app crashed again. 🙄',
  'review_key': 'api:A04'},
 {'ingestion_id': 'I012',
  'source': 'api',
  'source_record_id': 'A05',
  'business_area': 'e-commerce',
  'language': 'en',
  'rating': 3,
  'raw_text': 'The refund was quick; delivery was late.',
  'review_key': 'api:A05'},
 {'ingestion_id': 'I013',
  'source': 'api',
  'source_record_id': 'A06',
  'business_area': 'e-commerce',
  'language': 'en-ur-Latn',
  'rating': 3,
  'raw_text': 'Product acha hai but delivery bohat slow thi.',
  'review_key': 'api:A06'},
 {'ingestion_id': 'I014',
  'source': 'api',
  'source_record_id': 'A07',
  'business_area': 'retail',
  'language': 'en',
  'rating': None,
  'raw_text': None,
  'review_key': 'api:A07'},
 {'ingestion_id': 'I015',
  'source': 'api',
  'source_record_id': 'A08',
  'business_area': 'hospitality',
  'language': 'en',
  'rating': None,
  'raw_text': '   ',
  'review_key': 'api:A08'},
 {'ingestion_id': 'I016',
  'source': 'api',
  'source_record_id': 'A09',
  'business_area': 'banking',
  'language': 'en',
  'rating': 2,
  'raw_text': 'Email learner@example.invalid: refund not received.',
  'review_key': 'api:A09'}]

input_path = output_dir / "week10_customer_reviews.csv"  # Look for the previous week's handoff.
if input_path.exists():  # Prefer an available exported input.
    raw = pd.read_csv(input_path, dtype={"ingestion_id": "string", "source_record_id": "string"}, encoding="utf-8")  # Preserve textual identifiers.
    input_origin = "Week 10 CSV"  # Record the chosen source.
else:  # Allow independent use without a separate dataset attachment.
    raw = pd.DataFrame(embedded_records)  # Load the same synthetic records from this notebook.
    input_origin = "Embedded Week 10 teaching dataset"  # Make fallback use explicit.
required = {"ingestion_id", "source", "source_record_id", "business_area", "language", "rating", "raw_text"}  # Define the input contract.
if not required.issubset(raw.columns):  # Validate the schema before preprocessing.
    raise ValueError(f"Missing required columns: {sorted(required - set(raw.columns))}")  # Report a concrete input problem.
raw = raw.copy(deep=True)  # Keep the loaded input as the unchanged reference.
print("Input:", input_origin, "| Collected rows:", len(raw))  # Show provenance and size.
display(raw[["ingestion_id", "source_record_id", "language", "raw_text"]].head(5))  # Preview raw evidence.

Input: Week 10 CSV | Collected rows: 16


,ingestion_id,source_record_id,language,raw_text
0,I001,W01,en,"Delivery was late, but the product is good."
1,I002,W02,en,<p>Excellent support! 😊</p>
2,I003,W03,en,I can't get a refund. Support isn't helpful.
3,I004,W04,en,"Room was clean, but Wi-Fi was slow."
4,I005,W05,en,Checkout queue was long; staff were helpful.


### **Step 2 — Audit Missing Text and Repeated Collection**

The same `source + source_record_id` denotes one review. Identical repeated collection can be excluded from counts. Conflicting copies with the same key are stopped for review; the notebook does not silently choose between different versions.

In [ ]:
audit = raw.copy(deep=True)  # Add derived columns without changing the original input.
audit["review_key"] = audit["source"].astype(str) + ":" + audit["source_record_id"].astype(str)  # Rebuild source-scoped identity.
audit["missing_text"] = audit["raw_text"].isna()  # Identify missing values.
audit["blank_text"] = audit["raw_text"].fillna("").astype(str).str.strip().eq("") & ~audit["missing_text"]  # Keep blanks separate.
payload_fields = ["business_area", "language", "rating", "raw_text"]  # Compare substantive fields for duplicate keys.
conflicting = audit.groupby("review_key")[payload_fields].nunique(dropna=False).gt(1).any(axis=1)  # Detect inconsistent repeated records.
if conflicting.any():  # Require a deliberate decision for conflicting versions.
    raise ValueError(f"Conflicting review keys require review: {conflicting[conflicting].index.tolist()}")  # Preserve the ambiguity.
audit["duplicate_record"] = audit.duplicated("review_key", keep="first")  # Mark repeated collection after checking conflicts.
audit["status"] = "eligible"  # Start with a candidate document.
audit.loc[audit["language"].ne("en"), "status"] = "separate_language_workflow"  # Preserve incompatible language outside English counts.
audit.loc[audit["blank_text"], "status"] = "blank_text"  # Record a blank-text exclusion.
audit.loc[audit["missing_text"], "status"] = "missing_text"  # Record a missing-text exclusion.
audit.loc[audit["duplicate_record"], "status"] = "duplicate_collection"  # Count a source review only once.
display(audit["status"].value_counts().rename_axis("Status").reset_index(name="Records"))  # Display disjoint audit outcomes.

,Status,Records
0,eligible,12
1,duplicate_collection,1
2,separate_language_workflow,1
3,missing_text,1
4,blank_text,1


### **Step 3 — Define the Meaning-Preserving Cleaning Policy**

The function parses HTML, applies NFKC, replaces URLs and emails, expands the sample contractions, maps selected emojis, and normalizes whitespace. It preserves punctuation until tokenization, retains quantities, and uses only a reviewed correction for `sooo`.

**Limit:** This is a small, transparent teaching policy. The URL/email patterns are not a comprehensive privacy filter; its contraction, slang, and emoji dictionaries do not cover all language.

In [ ]:
contractions = {"can't": "can not", "won't": "will not", "isn't": "is not", "don't": "do not", "didn't": "did not", "hasn't": "has not", "doesn't": "does not"}  # Expand known forms before punctuation handling.
emoji_map = {"😊": " smile_emoji ", "😡": " angry_emoji ", "🙄": " eye_roll_emoji "}  # Preserve selected symbols as descriptive markers.
def normalize_text(value):  # Apply one consistent policy to each document.
    if pd.isna(value):  # Handle an absent comment before string conversion.
        return ""  # Avoid introducing a literal nan token.
    text = unicodedata.normalize("NFKC", str(value))  # Normalize compatibility forms in the teaching scope.
    if re.search(r"</?[A-Za-z][^>]*>", text):  # Parse values containing apparent HTML markup.
        parsed = BeautifulSoup(text, "html.parser")  # Use an HTML parser instead of stripping tags by regex.
        for element in parsed(["script", "style"]):  # Identify non-review executable or style content.
            element.decompose()  # Remove those elements before extracting text.
        text = parsed.get_text(" ", strip=True)  # Keep spaces between adjacent text elements.
    text = text.replace("’", "'").replace("‘", "'")  # Standardize apostrophes for known contractions.
    text = re.sub(r"https?://[^\s<>]+|www\.[^\s<>]+", " URL_TOKEN ", text)  # Replace links with a marker for this controlled sample.
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", " EMAIL_TOKEN ", text)  # Replace email spans in the analytical view.
    text = text.lower()  # Standardize case for English-oriented phrase counts.
    for short, expanded in contractions.items():  # Apply only the documented contraction mappings.
        text = re.sub(r"\b" + re.escape(short) + r"\b", expanded, text)  # Preserve negation with boundary-aware replacement.
    for symbol, marker in emoji_map.items():  # Preserve the selected emoji signals.
        text = text.replace(symbol, marker)  # Keep marker boundaries explicit.
    text = re.sub(r"\bsooo\b", "so", text)  # Normalize one known repeated-character variant.
    text = re.sub(r"(?<=\d),(?=\d{3}(?:\D|$))", "", text)  # Normalize the fixture's English thousands separator.
    return re.sub(r"\s+", " ", text).strip()  # Collapse incidental whitespace without deleting punctuation.
audit["normalized_text"] = audit["raw_text"].map(normalize_text)  # Create a derived representation for every row.
display(audit.loc[[1, 2, 7, 8], ["raw_text", "normalized_text"]])  # Inspect HTML, contractions, amounts, and emoji examples.

,raw_text,normalized_text
1,<p>Excellent support! 😊</p>,excellent support! smile_emoji
2,I can't get a refund. Support isn't helpful.,i can not get a refund. support is not helpful.
7,"Payment failed twice! PKR 2,500 was deducted.",payment failed twice! pkr 2500 was deducted.
8,Delivery was sooo LATE!!! 😡 See https://example.com/order/42,delivery was so late!!! angry_emoji see url_token


### **Step 4 — Compare Tokenizers and Build Sentence-Aware Tokens**

NLTK's Treebank and Tweet tokenizers demonstrate different handling of contractions, punctuation, and emoji. The corpus uses a documented regex view, after normalization, so its behavior remains easy to inspect.

Sentence splitting below is appropriate for these controlled examples, not arbitrary abbreviations or multilingual punctuation. The code keeps an independent token list for each sentence.

In [ ]:
sample = "I can't pay!!! 😡"  # Use the same input for a fair tokenizer comparison.
display(pd.DataFrame({"Method": ["Whitespace", "NLTK Treebank", "NLTK Tweet"],  # Name each tokenization policy.
                      "Tokens": [sample.split(), TreebankWordTokenizer().tokenize(sample), TweetTokenizer().tokenize(sample)]}))  # Show actual differences.
token_pattern = re.compile(r"\d+(?:\.\d+)?|[^\W\d]+(?:\d+)?", flags=re.UNICODE)  # Keep words, markers, and numerical forms.
def sentence_tokens(text):  # Preserve local boundaries for phrase generation.
    sentences = re.split(r"[.!?]+(?:\s+|$)", text)  # Split the sample's punctuation-delimited sentences.
    return [tokens for sentence in sentences if (tokens := token_pattern.findall(sentence))]  # Keep non-empty token sequences.
audit["sentence_tokens"] = audit["normalized_text"].map(sentence_tokens)  # Store tokens by sentence.
audit["tokens"] = audit["sentence_tokens"].map(lambda sentences: [token for sentence in sentences for token in sentence])  # Flatten only within each document for word statistics.
display(audit.loc[[2, 5], ["review_key", "normalized_text", "sentence_tokens"]])  # Inspect contractions and negative wording.

,Method,Tokens
0,Whitespace,"[I, can't, pay!!!, 😡]"
1,NLTK Treebank,"[I, ca, n't, pay, !, !, !, 😡]"
2,NLTK Tweet,"[I, can't, pay, !, !, !, 😡]"


,review_key,normalized_text,sentence_tokens
2,web:W03,i can not get a refund. support is not helpful.,"[[i, can, not, get, a, refund], [support, is, not, helpful]]"
5,web:W06,delivery was not late. great service!,"[[delivery, was, not, late], [great, service]]"


### **Step 5 — Create a Separate Stop-Word View**

Use a deliberately small, auditable list. This filtered view can support later word-count exploration; it does not replace the original tokens and is not used to form n-grams.

In [ ]:
stop_words = {"a", "an", "the", "is", "was", "were", "am", "are", "be", "been", "to", "of", "and", "i", "my", "it", "this", "that"}  # Define task-specific candidates explicitly.
protected_words = {"not", "no", "never", "but", "very"}  # Keep negation, contrast, and selected intensity signals.
stop_words = stop_words - protected_words  # Ensure protected terms cannot be removed by this list.
audit["filtered_tokens"] = audit["tokens"].map(lambda tokens: [word for word in tokens if word not in stop_words])  # Create an optional derived view.
display(audit.loc[[2, 5], ["tokens", "filtered_tokens"]])  # Confirm that negation survives.

,tokens,filtered_tokens
2,"[i, can, not, get, a, refund, support, is, not, helpful]","[can, not, get, refund, support, not, helpful]"
5,"[delivery, was, not, late, great, service]","[delivery, not, late, great, service]"


### **Step 6 — Compare Stems and Lemmas**

NLTK supplies the Porter and English Snowball stemmers. Simplemma supplies dictionary-based English lemmatization with bundled lexical data. These are three alternative outputs; the code does not stem first and then lemmatize a stem.

**Read the output:** `deliveri` is a stem, not a spelling error to “correct.” A lemma such as `good` for `better` reflects a lexical mapping; context may require a different interpretation.

In [ ]:
porter = PorterStemmer()  # Initialize the default NLTK Porter implementation.
snowball = SnowballStemmer("english")  # Choose the language explicitly.
words = ["deliveries", "delivered", "running", "better", "children", "payments"]  # Select contrasting examples.
comparison = pd.DataFrame({"Word": words,  # Keep the original form for interpretation.
                           "Porter stem": [porter.stem(word) for word in words],  # Apply one stemming alternative.
                           "Snowball stem": [snowball.stem(word) for word in words],  # Apply the second alternative.
                           "Simplemma lemma": [lemmatize(word, lang="en") for word in words]})  # Apply dictionary-based lemmatization independently.
display(comparison)  # Compare actual outputs rather than assuming every method agrees.
audit["lemmas"] = [  # Retain a lemma view only where the English policy applies.
    [lemmatize(token, lang="en") for token in tokens] if language == "en" else []  # Avoid applying English morphology to mixed-language records.
    for tokens, language in zip(audit["tokens"], audit["language"])  # Keep output rows aligned with input rows.
]  # Finish the derived lemma column.

,Word,Porter stem,Snowball stem,Simplemma lemma
0,deliveries,deliveri,deliveri,delivery
1,delivered,deliv,deliv,deliver
2,running,run,run,run
3,better,better,better,good
4,children,children,children,child
5,payments,payment,payment,payment


### **Step 7 — Build the Eligible English Corpus**

Apply the previously recorded decisions. The mixed-language review stays in the full audit with its normalized text and tokens; it is excluded only from the English corpus. Empty token lists require a visible exclusion reason.

In [ ]:
empty_tokens = audit["tokens"].map(len).eq(0)  # Detect any document emptied by the tokenization policy.
audit.loc[audit["status"].eq("eligible") & empty_tokens, "status"] = "empty_after_processing"  # Report newly empty candidates.
corpus = audit.loc[audit["status"].eq("eligible")].copy()  # Select accepted, unique English records.
corpus["token_count"] = corpus["tokens"].map(len)  # Measure document length under the stated tokenization policy.
vocabulary = {token for tokens in corpus["tokens"] for token in tokens}  # Count distinct normalized tokens.
stats = pd.DataFrame({"Measure": ["Collected records", "Eligible English documents", "Token occurrences", "Vocabulary size"],  # Name distinct size concepts.
                      "Value": [len(audit), len(corpus), int(corpus["token_count"].sum()), len(vocabulary)]})  # Calculate from executed results.
display(stats)  # Display a compact corpus summary.
display(audit.loc[~audit["status"].eq("eligible"), ["review_key", "language", "status"]])  # Make exclusions inspectable.

,Measure,Value
0,Collected records,16
1,Eligible English documents,12
2,Token occurrences,83
3,Vocabulary size,55


,review_key,language,status
6,web:W01,en,duplicate_collection
12,api:A06,en-ur-Latn,separate_language_workflow
13,api:A07,en,missing_text
14,api:A08,en,blank_text


### **Step 8 — Generate Unigrams, Bigrams, and Trigrams**

Use full normalized tokens within each sentence. Count occurrences and the number of eligible reviews containing each phrase. A phrase that appears twice in the same review contributes two occurrences but one review mention.

In [ ]:
def make_ngrams(tokens, n):  # Form contiguous sequences inside one token list.
    return [tuple(tokens[i:i+n]) for i in range(max(len(tokens) - n + 1, 0))]  # Respect the n-gram position formula.
def phrase_counts(documents, n):  # Aggregate sentence-bounded phrases across documents.
    occurrences, document_frequency = Counter(), Counter()  # Separate repetition from review coverage.
    for sentences in documents:  # Process one review at a time.
        phrases = [phrase for sentence in sentences for phrase in make_ngrams(sentence, n)]  # Do not cross sentence boundaries.
        occurrences.update(phrases)  # Count every occurrence.
        document_frequency.update(set(phrases))  # Count each phrase once per review.
    rows = [{"phrase": " ".join(phrase), "occurrences": count, "reviews": document_frequency[phrase]} for phrase, count in occurrences.items()]  # Create readable rows.
    return pd.DataFrame(rows, columns=["phrase", "occurrences", "reviews"]).sort_values(["occurrences", "phrase"], ascending=[False, True]).reset_index(drop=True)  # Sort deterministically.
phrase_tables = {n: phrase_counts(corpus["sentence_tokens"], n) for n in [1, 2, 3]}  # Compute the three requested n-gram sizes.
for n, table in phrase_tables.items():  # Show bounded comparisons instead of a raw corpus dump.
    print(f"Top {n}-grams — {len(corpus)} eligible English reviews")  # State the denominator and n.
    display(table.head(6))  # Display the most frequent examples with occurrence and review counts.

Top 1-grams — 12 eligible English reviews


,phrase,occurrences,reviews
0,was,9,7
1,not,5,4
2,delivery,4,4
3,late,4,4
4,refund,3,3
5,the,3,3


Top 2-grams — 12 eligible English reviews


,phrase,occurrences,reviews
0,delivery was,4,4
1,was late,2,2
2,2 hours,1,1
3,2500 was,1,1
4,a refund,1,1
5,angry_emoji see,1,1


Top 3-grams — 12 eligible English reviews


,phrase,occurrences,reviews
0,delivery was late,2,2
1,2500 was deducted,1,1
2,angry_emoji see url_token,1,1
3,app crashed again,1,1
4,battery lasts only,1,1
5,but the product,1,1


### **Step 9 — Inspect Phrase Evidence Before Naming a Pain Point**

Retrieve supporting reviews for both `delivery was` and `not late`. The second phrase reminds us that mentioning delay-related vocabulary is not the same as reporting a delay. The function matches token phrases under the same policy used to count them.

In [ ]:
def contains_phrase(sentences, phrase):  # Check exact adjacency under the corpus token policy.
    target = tuple(phrase.split())  # Use space-separated normalized words for this exercise.
    return any(target in make_ngrams(sentence, len(target)) for sentence in sentences)  # Search within each sentence only.
for phrase in ["delivery was", "not late"]:  # Contrast a common topic phrase with a negated expression.
    matches = corpus["sentence_tokens"].map(lambda sentences: contains_phrase(sentences, phrase))  # Locate supporting records.
    print(f"Evidence for: {phrase}")  # Identify the phrase being inspected.
    display(corpus.loc[matches, ["review_key", "raw_text"]])  # Read raw evidence before interpreting customer experience.

Evidence for: delivery was


,review_key,raw_text
0,web:W01,"Delivery was late, but the product is good."
5,web:W06,Delivery was not late. Great service!
8,api:A02,Delivery was sooo LATE!!! 😡 See https://example.com/order/42
11,api:A05,The refund was quick; delivery was late.


Evidence for: not late


,review_key,raw_text
5,web:W06,Delivery was not late. Great service!


### **Step 10 — Validate Meaning, Boundaries, and Coverage**

These checks verify concrete risks: negation loss, merged HTML words, destroyed quantities, invented cross-sentence phrases, repeated review IDs, and changes to raw data. They do not validate a sentiment model; none has been trained here.

In [ ]:
assert "not" in sentence_tokens(normalize_text("The service isn't good."))[0]  # Verify contraction expansion preserves negation.
assert "not" not in stop_words  # Verify stop-word policy protects negation.
assert normalize_text("<b>slow</b><i>delivery</i>") == "slow delivery"  # Prevent adjacent HTML tags from merging words.
assert "2500" in normalize_text("PKR 2,500 deducted")  # Retain the meaningful amount in the documented format.
assert "angry_emoji" in normalize_text("Late 😡")  # Retain the selected emotion marker.
assert "email_token" in normalize_text("Contact learner@example.invalid")  # Verify the controlled email example is replaced.
assert not contains_phrase(sentence_tokens("refund pending. delivery late."), "pending delivery")  # Prevent cross-sentence adjacency.
assert corpus["review_key"].is_unique and corpus["tokens"].map(len).gt(0).all()  # Validate corpus identity and nonempty documents.
assert audit["raw_text"].equals(raw["raw_text"])  # Confirm the original evidence is unchanged.
assert audit["normalized_text"].map(normalize_text).equals(audit["normalized_text"])  # Confirm normalized text is idempotent for these records.
print("Passed: meaning-critical examples, boundaries, record identity, raw-text preservation, and idempotence.")  # Report the checks actually performed.

Passed: meaning-critical examples, boundaries, record identity, raw-text preservation, and idempotence.


### **Step 11 — Export the Corpus and Processing Audit**

The full audit preserves every collection row and exclusion reason. The corpus contains only eligible English documents. Nested token lists are serialized as JSON strings inside CSV fields and verified on reload.

In [ ]:
list_columns = ["sentence_tokens", "tokens", "filtered_tokens", "lemmas"]  # Identify values requiring explicit CSV serialization.
def export_table(frame, filename):  # Save a reusable table without changing the in-memory data.
    saved = frame.copy()  # Work on a separate export view.
    for column in list_columns:  # Serialize each list-valued field consistently.
        saved[column] = saved[column].map(lambda value: json.dumps(value, ensure_ascii=False))  # Use valid JSON rather than Python repr.
    saved["policy_version"] = policy_version  # Attach the transformation-policy identity.
    saved.to_csv(output_dir / filename, index=False, encoding="utf-8")  # Preserve Unicode in a portable CSV.
export_table(audit, "week11_processing_audit.csv")  # Save all records and exclusion reasons.
export_table(corpus, "week11_cleaned_english_corpus.csv")  # Save accepted English documents.
all_phrases = pd.concat([table.assign(n=n) for n, table in phrase_tables.items()], ignore_index=True)  # Combine phrase summaries with their n value.
all_phrases.to_csv(output_dir / "week11_ngram_counts.csv", index=False, encoding="utf-8")  # Save exact counts.
policy = {"policy_version": policy_version, "input_origin": input_origin, "synthetic_teaching_data": True, "language_scope": "English corpus; other languages retained in audit", "stop_words": sorted(stop_words), "ngram_boundaries": "within each sentence and document, before stop-word removal", "versions": {package: version(package) for package in ["pandas", "beautifulsoup4", "nltk", "simplemma"]}}  # Document reproducibility choices.
(output_dir / "week11_preprocessing_policy.json").write_text(json.dumps(policy, ensure_ascii=False, indent=2), encoding="utf-8")  # Export policy and environment versions.
reloaded = pd.read_csv(output_dir / "week11_cleaned_english_corpus.csv", encoding="utf-8")  # Verify the exported corpus can be reloaded.
assert reloaded["tokens"].map(json.loads).tolist() == corpus["tokens"].tolist()  # Check list serialization and Unicode survive the round trip.
print(f"Saved audit, English corpus ({len(corpus)} documents), n-gram counts, and preprocessing policy to {output_dir}.")  # Confirm final handoff.

Saved audit, English corpus (12 documents), n-gram counts, and preprocessing policy to ba3002_text_mining_data.
